# PROYECTO DE APLICACIÓN RECOMMENDATION SYSTEMS

## Carga de datos

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
# Descargamos el dataset MovieLens
!wget https://files.grouplens.org/datasets/movielens/ml-latest-small.zip

# Descomprimimos el archivo
!unzip -o ml-latest-small.zip

--2026-09-26 06:19:45--  https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 978202 (955K) [application/zip]
Saving to: ‘ml-latest-small.zip’

ml-latest-small.zip 100%[===================>] 955.28K  --.-KB/s    in 0.1s    

2026-09-26 06:19:45 (7.28 MB/s) - ‘ml-latest-small.zip’ saved [978202/978202]

Archive:  ml-latest-small.zip
   creating: ml-latest-small/
  inflating: ml-latest-small/links.csv  
  inflating: ml-latest-small/tags.csv  
  inflating: ml-latest-small/ratings.csv  
  inflating: ml-latest-small/README.txt  
  inflating: ml-latest-small/movies.csv  


In [5]:
# Cargamos los archivos del dataset MovieLens
movies = pd.read_csv("ml-latest-small/movies.csv")
ratings = pd.read_csv("ml-latest-small/ratings.csv")

# Mostramos las primeras filas
print("Películas:")
display(movies.head())

print("Valoraciones:")
display(ratings.head())

Películas:


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


Valoraciones:


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [6]:
# Comprobamos el tamaño de los datasets
print("Tamaño de movies:", movies.shape)
print("Tamaño de ratings:", ratings.shape)

# Comprobamos las columnas
print("\nColumnas de movies:")
print(movies.columns)

print("\nColumnas de ratings:")
print(ratings.columns)

# Comprobamos si existen valores nulos
print("\nValores nulos en movies:")
print(movies.isnull().sum())

print("\nValores nulos en ratings:")
print(ratings.isnull().sum())

Tamaño de movies: (9742, 3)
Tamaño de ratings: (100836, 4)

Columnas de movies:
Index(['movieId', 'title', 'genres'], dtype='object')

Columnas de ratings:
Index(['userId', 'movieId', 'rating', 'timestamp'], dtype='object')

Valores nulos en movies:
movieId    0
title      0
genres     0
dtype: int64

Valores nulos en ratings:
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64


## Construcción de la matriz de calificaciones

In [7]:
# Creamos la matriz usuario-película
matriz_ratings = ratings.pivot_table(
    index="userId",
    columns="movieId",
    values="rating"
)

# Mostramos las primeras filas de la matriz
display(matriz_ratings.head())

# Mostramos las dimensiones
print("Dimensiones de la matriz:", matriz_ratings.shape)

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,NaN,4.0,NaN,NaN,4.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Dimensiones de la matriz: (610, 9724)


In [8]:
# Sustituimos los valores NaN por 0
matriz_ratings = matriz_ratings.fillna(0)

# Comprobamos el resultado
display(matriz_ratings.head())

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,0.0,4.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Calcular la similitud entre películas

In [9]:
# Trasponemos la matriz para tener las películas en las filas
matriz_peliculas = matriz_ratings.T

# Calculamos la similitud del coseno entre las películas
similitud_peliculas = cosine_similarity(matriz_peliculas)

# Comprobamos las dimensiones de la matriz de similitud
print("Dimensiones:", similitud_peliculas.shape)

Dimensiones: (9724, 9724)


In [10]:
# Convertimos la matriz de similitud en un DataFrame
similitud_df = pd.DataFrame(
    similitud_peliculas,
    index=matriz_peliculas.index,
    columns=matriz_peliculas.index
)

# Mostramos una pequeña parte de la matriz
display(similitud_df.iloc[:5, :5])

movieId,1,2,3,4,5
movieId,,,,,
1,1.000000,0.410562,0.296917,0.035573,0.308762
2,0.410562,1.000000,0.282438,0.106415,0.287795
3,0.296917,0.282438,1.000000,0.092406,0.417802
4,0.035573,0.106415,0.092406,1.000000,0.188376
5,0.308762,0.287795,0.417802,0.188376,1.000000


## Implementación del Motor de Recomendación

In [11]:
# Creamos una función para recomendar películas similares
def recomendar_peliculas(titulo, n=5):

    # Buscamos el ID de la película a partir de su título
    pelicula = movies[movies["title"] == titulo]

    # Comprobamos que la película existe
    if pelicula.empty:
        return "Película no encontrada"

    movie_id = pelicula.iloc[0]["movieId"]

    # Obtenemos las similitudes y las ordenamos de mayor a menor
    similares = similitud_df[movie_id].sort_values(ascending=False)

    # Excluimos la propia película y seleccionamos las N más similares
    similares = similares.iloc[1:n+1]

    # Obtenemos los títulos de las películas recomendadas
    recomendaciones = movies[movies["movieId"].isin(similares.index)]

    return recomendaciones[["movieId", "title"]]

## Ejemplo de Uso

In [13]:
# Probamos el motor de recomendación con una película
titulo_ejemplo = "Toy Story (1995)"

print("Películas similares a:", titulo_ejemplo)

# Mostramos las 5 recomendaciones
display(recomendar_peliculas(titulo_ejemplo, n=5))

Películas similares a: Toy Story (1995)


,movieId,title
224,260,Star Wars: Episode IV - A New Hope (1977)
314,356,Forrest Gump (1994)
418,480,Jurassic Park (1993)
615,780,Independence Day (a.k.a. ID4) (1996)
2355,3114,Toy Story 2 (1999)
